# Méta-analyse : l'évolution des grands modèles de langage (LLM)
### Étude comparative de cinq articles fondateurs, de l'architecture Transformer au scaling massif

## 1. Introduction

Depuis 2017, le traitement automatique du langage naturel (NLP) a connu une transformation profonde, portée par une succession d'innovations architecturales et méthodologiques qui ont donné naissance à ce que l'on appelle aujourd'hui les grands modèles de langage (Large Language Models, LLM). Ce rapport propose une méta-analyse de cinq articles considérés comme fondateurs de ce domaine : *Attention Is All You Need* (Vaswani et al., 2017), *BERT* (Devlin et al., 2019), l'article fondateur de la famille *GPT* (Radford et al., 2018), *GPT-3* (Brown et al., 2020) et *PaLM* (Chowdhery et al., 2022).

**Objectifs de l'analyse.** Il s'agit ici non pas de présenter chaque article isolément, mais de retracer le fil conducteur qui les relie : comment une architecture unique, le Transformer, a servi de socle à des stratégies de pré-entraînement de plus en plus variées (bidirectionnelle, autoregressive, few-shot), puis à une course au passage à l'échelle (*scaling*) qui a progressivement transformé la nature même des capacités de ces modèles. L'analyse vise à identifier les problèmes de recherche traités par chaque article, les solutions techniques proposées, les données utilisées pour l'entraînement et l'évaluation, ainsi que les résultats obtenus, avant de dégager les tendances communes, les limites partagées et les pistes de recherche qui en découlent.

**Thème commun.** Les cinq articles retracent une trajectoire cohérente : l'abandon des architectures récurrentes au profit de l'attention (Transformer), l'exploration de deux grandes familles d'utilisation de cette architecture — l'encodage bidirectionnel (BERT) et la génération autorégressive (GPT) —, puis la démonstration que l'augmentation de la taille des modèles et des corpus d'entraînement (GPT-3, PaLM) fait émerger des capacités nouvelles, notamment l'apprentissage en contexte (*in-context learning*), sans qu'il soit nécessaire de ré-entraîner le modèle pour chaque tâche.

## 2. Articles sélectionnés

| # | Titre | Auteurs (principaux) | Venue |
|---|-------|----------------------|-------|
| 1 | Attention Is All You Need | Vaswani et al. | NeurIPS 2017 |
| 2 | BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding | Devlin et al. | NAACL 2019 |
| 3 | Improving Language Understanding by Generative Pre-Training (GPT) | Radford et al. | OpenAI, 2018 |
| 4 | Language Models are Few-Shot Learners (GPT-3) | Brown et al. | NeurIPS 2020 |
| 5 | PaLM: Scaling Language Modeling with Pathways | Chowdhery et al. | arXiv:2204.02311, 2022 |

## 3. Résumés détaillés des articles

### 3.1 Attention Is All You Need (Vaswani et al., 2017)

**Problème de recherche.** Jusqu'en 2017, les meilleurs modèles de traduction automatique et de modélisation de séquences reposaient sur des réseaux récurrents (RNN, LSTM) ou convolutifs. Ces architectures traitent les séquences token par token, ce qui empêche toute parallélisation au sein d'une même séquence pendant l'entraînement et complique la capture de dépendances entre mots éloignés dans le texte.

**Solution proposée.** Les auteurs introduisent le *Transformer*, une architecture encodeur-décodeur qui se passe entièrement de récurrence et de convolution au profit de mécanismes d'attention, en particulier l'auto-attention multi-têtes (*multi-head self-attention*). Chaque position de la séquence peut ainsi « regarder » directement toutes les autres positions, quelle que soit leur distance, et l'ensemble du calcul devient parallélisable. Un encodage positionnel est ajouté pour compenser l'absence de récurrence et préserver l'information sur l'ordre des mots.

**Données utilisées.** Le modèle est évalué sur des tâches de traduction automatique : WMT 2014 anglais-allemand (environ 4,5 millions de paires de phrases) et WMT 2014 anglais-français (environ 36 millions de paires de phrases).

**Résultats principaux.** Le Transformer établit un nouvel état de l'art en traduction automatique (28,4 BLEU en anglais-allemand, 41,8 BLEU en anglais-français) tout en réduisant considérablement le temps et le coût d'entraînement grâce à la parallélisation, par rapport aux architectures récurrentes ou convolutives antérieures.

### 3.2 BERT (Devlin et al., 2019)

**Problème de recherche.** Les modèles de langage pré-entraînés existants (comme GPT) sont unidirectionnels : ils ne considèrent le contexte que de gauche à droite, ce qui limite leur capacité à modéliser finement le sens d'un mot en fonction de tout son contexte, y compris les mots qui le suivent — une limite pénalisante pour des tâches de compréhension (réponse à des questions, inférence, etc.).

**Solution proposée.** BERT introduit un pré-entraînement bidirectionnel profond fondé sur deux tâches auto-supervisées : le *Masked Language Model* (MLM), qui masque aléatoirement une partie des tokens d'entrée et demande au modèle de les reconstituer à partir du contexte des deux côtés, et la *Next Sentence Prediction* (NSP), qui entraîne le modèle à prédire si deux phrases se suivent réellement dans le texte d'origine. L'architecture repose uniquement sur l'encodeur du Transformer. Une fois pré-entraîné, le modèle peut être affiné (*fine-tuné*) rapidement sur des tâches spécifiques en ajoutant une simple couche de sortie.

**Données utilisées.** Le pré-entraînement combine le BooksCorpus (environ 800 millions de mots) et l'intégralité de la Wikipédia anglaise (environ 2,5 milliards de mots). L'évaluation s'appuie sur les benchmarks GLUE, SQuAD (v1.1 et v2.0) et SWAG.

**Résultats principaux.** BERT établit un nouvel état de l'art sur onze tâches de NLP, avec un score GLUE de 80,5 % et des gains significatifs sur SQuAD, démontrant que la bidirectionnalité profonde du pré-entraînement apporte un avantage déterminant par rapport aux approches unidirectionnelles.

### 3.3 GPT — Generative Pre-Training (Radford et al., 2018)

**Problème de recherche.** La plupart des approches de NLP supervisées nécessitent de grandes quantités de données annotées manuellement pour chaque tâche, une ressource rare et coûteuse à produire, alors que le texte non annoté est disponible en abondance.

**Solution proposée.** Les auteurs proposent une approche en deux temps : un pré-entraînement génératif non supervisé d'un modèle de langage sur un grand corpus de texte non annoté (prédiction du mot suivant, architecture Transformer *decoder-only*), suivi d'un fine-tuning discriminatif supervisé sur chaque tâche cible, en réutilisant les représentations apprises. Cette méthode permet de tirer parti du texte non labellisé pour réduire la dépendance aux données annotées spécifiques à une tâche.

**Données utilisées.** Le pré-entraînement s'appuie sur le BooksCorpus (environ 7 000 livres non publiés, choisis pour leurs passages narratifs longs et cohérents). Le fine-tuning est ensuite réalisé sur des jeux de données variés couvrant l'inférence en langage naturel, la classification, la similarité de phrases et la réponse à des questions.

**Résultats principaux.** Le modèle améliore l'état de l'art sur 9 des 12 tâches étudiées, démontrant que le pré-entraînement génératif non supervisé, combiné à un fine-tuning léger, constitue une stratégie efficace et généralisable pour le NLP.

### 3.4 GPT-3 (Brown et al., 2020)

**Problème de recherche.** Même les modèles pré-entraînés comme BERT ou GPT nécessitent un fine-tuning sur des milliers d'exemples annotés pour chaque nouvelle tâche, ce qui reste coûteux et limite leur capacité à s'adapter rapidement à des tâches nouvelles ou rares, contrairement à l'humain qui apprend souvent à partir de quelques exemples seulement.

**Solution proposée.** Les auteurs démontrent qu'en augmentant massivement la taille du modèle (175 milliards de paramètres, contre 1,5 milliard pour la plus grande version de GPT-2) et du corpus d'entraînement, un modèle purement autorégressif peut réaliser des tâches en *zero-shot*, *one-shot* ou *few-shot* : il suffit de lui fournir quelques exemples dans l'invite (le *prompt*), sans mettre à jour ses poids, pour qu'il généralise à une nouvelle tâche. C'est ce que les auteurs nomment l'apprentissage en contexte (*in-context learning*).

**Données utilisées.** L'entraînement combine une version filtrée de Common Crawl, WebText2, deux corpus de livres (Books1 et Books2) et Wikipédia, pour un total d'environ 300 milliards de tokens traités durant l'entraînement.

**Résultats principaux.** GPT-3 obtient des performances compétitives, voire état de l'art, sur de nombreuses tâches de NLP en few-shot sans aucun fine-tuning, mais montre encore des limites sur certaines tâches nécessitant un raisonnement logique ou arithmétique complexe, ainsi que des biais hérités des données d'entraînement issues du web.

### 3.5 PaLM (Chowdhery et al., 2022)

**Problème de recherche.** Le passage à l'échelle des LLM au-delà de la centaine de milliards de paramètres se heurte à des limites d'infrastructure : l'entraînement distribué sur des milliers d'accélérateurs devient difficile à orchestrer efficacement, ce qui freine l'exploration de modèles encore plus grands.

**Solution proposée.** Les auteurs présentent PaLM, un modèle Transformer *decoder-only* de 540 milliards de paramètres, entraîné grâce au système *Pathways*, qui permet un entraînement efficace et hautement parallélisé sur plusieurs pods de TPU simultanément. Le modèle intègre également plusieurs améliorations architecturales (activations SwiGLU, couches parallèles, attention multi-requêtes, encodages positionnels RoPE) visant à améliorer l'efficacité d'entraînement et d'inférence.

**Données utilisées.** Le corpus d'entraînement, multilingue, mélange des pages web filtrées, des livres, la Wikipédia, des articles de presse et du code source issu de GitHub, pour un total d'environ 780 milliards de tokens.

**Résultats principaux.** PaLM établit un nouvel état de l'art en few-shot sur un grand nombre de benchmarks de langage et de raisonnement, et met en évidence des capacités émergentes de raisonnement par chaînage d'étapes (*chain-of-thought prompting*), certaines performances dépassant même les scores humains moyens sur des tests de compréhension du langage, ce qui illustre l'apparition de capacités qualitativement nouvelles avec l'augmentation de l'échelle.

## 4. Tableau comparatif

Le code ci-dessous construit un tableau de synthèse rassemblant, pour chaque article, les contributions clés, l'architecture du modèle et ses caractéristiques les plus marquantes, en complément des résumés détaillés ci-dessus.

In [ ]:
import pandas as pd

articles_data = [
    {
        'Titre': 'Attention Is All You Need',
        'Auteurs': 'Vaswani et al.',
        'Venue': 'NeurIPS 2017',
        'Contribution clé': "Architecture Transformer fondée uniquement sur l'attention, sans récurrence ni convolution.",
        'Architecture': 'Transformer (encodeur-décodeur)',
        'Caractéristiques marquantes': 'Auto-attention multi-têtes, parallélisable, meilleures dépendances longue distance.'
    },
    {
        'Titre': 'BERT',
        'Auteurs': 'Devlin et al.',
        'Venue': 'NAACL 2019',
        'Contribution clé': 'Pré-entraînement bidirectionnel profond via masquage de tokens (MLM) et prédiction de phrase suivante (NSP).',
        'Architecture': 'Transformer (encodeur seul)',
        'Caractéristiques marquantes': 'Bidirectionnalité profonde, fine-tuning léger, état de l\'art sur 11 tâches NLP.'
    },
    {
        'Titre': 'GPT (Generative Pre-Training)',
        'Auteurs': 'Radford et al.',
        'Venue': 'OpenAI, 2018',
        'Contribution clé': 'Pré-entraînement génératif non supervisé suivi de fine-tuning discriminatif par tâche.',
        'Architecture': 'Transformer (décodeur seul)',
        'Caractéristiques marquantes': 'Exploitation de texte non annoté, transfert efficace vers 9/12 tâches évaluées.'
    },
    {
        'Titre': 'GPT-3',
        'Auteurs': 'Brown et al.',
        'Venue': 'NeurIPS 2020',
        'Contribution clé': "Démonstration de l'apprentissage en contexte (few-shot) sans mise à jour des poids, à très grande échelle.",
        'Architecture': 'Transformer (décodeur seul), 175 milliards de paramètres',
        'Caractéristiques marquantes': 'Zero/one/few-shot learning, corpus ~300 milliards de tokens, limites en raisonnement complexe.'
    },
    {
        'Titre': 'PaLM',
        'Auteurs': 'Chowdhery et al.',
        'Venue': 'arXiv:2204.02311, 2022',
        'Contribution clé': "Entraînement efficace à très grande échelle via le système Pathways.",
        'Architecture': 'Transformer (décodeur seul), 540 milliards de paramètres',
        'Caractéristiques marquantes': 'Raisonnement en chaîne de pensée (chain-of-thought), corpus multilingue de 780 milliards de tokens.'
    },
]

llm_articles_df = pd.DataFrame(articles_data)
print(llm_articles_df.to_markdown(index=False))


### 4.1 Analyse comparative

Le tableau ci-dessus met en évidence trois axes d'évolution majeurs entre 2017 et 2022.

**Évolution architecturale.** Le Transformer originel adopte une structure encodeur-décodeur, pensée pour la traduction. BERT ne conserve que l'encodeur, optimisé pour la *compréhension* du langage (classification, extraction de réponses), tandis que GPT, GPT-3 et PaLM ne conservent que le décodeur, optimisé pour la *génération* de texte. Cette dernière configuration, plus simple à faire passer à l'échelle, est celle qui s'est finalement imposée comme standard pour les LLM modernes.

**Passage à l'échelle des paramètres et des données.** On observe une croissance quasi exponentielle du nombre de paramètres : de quelques dizaines de millions pour le Transformer original à 175 milliards pour GPT-3 puis 540 milliards pour PaLM, accompagnée d'une croissance comparable des corpus d'entraînement (de quelques milliards de mots pour BERT à plusieurs centaines de milliards de tokens pour GPT-3 et PaLM).

**Évolution du paradigme d'apprentissage.** Les stratégies d'apprentissage évoluent également : le Transformer est entraîné de façon supervisée sur une tâche unique (traduction) ; BERT et GPT introduisent le pré-entraînement auto-supervisé suivi d'un fine-tuning spécifique à la tâche ; GPT-3 et PaLM montrent enfin qu'à très grande échelle, le fine-tuning devient optionnel, le modèle pouvant s'adapter à une nouvelle tâche simplement à partir d'exemples fournis dans le contexte de l'invite. Cette dernière rupture change profondément la manière dont ces modèles sont utilisés en pratique.

Ces trois évolutions ne sont pas indépendantes : c'est bien la simplicité et la parallélisabilité de l'architecture Transformer, introduite en 2017, qui a rendu possible le passage à l'échelle observé dans les articles suivants, lequel a lui-même fait émerger de nouveaux comportements (apprentissage en contexte, raisonnement en chaîne de pensée).

## 5. Perspectives et réflexions

**Tendances observées.** L'ensemble des articles étudiés dessine une trajectoire claire vers le décodeur Transformer autorégressif comme architecture dominante, et vers le *scaling* — plus de paramètres, plus de données, plus de calcul — comme principal levier de progrès. Cette tendance s'accompagne de l'apparition de « capacités émergentes » (few-shot learning, raisonnement en chaîne de pensée) qui n'étaient ni explicitement conçues ni pleinement anticipées par les architectes des modèles, mais qui apparaissent progressivement à mesure que l'échelle augmente.

**Difficultés communes.** Plusieurs limites reviennent d'un article à l'autre. Le coût computationnel et énergétique de l'entraînement croît très rapidement avec la taille des modèles, posant des questions de soutenabilité et d'accessibilité (seuls quelques acteurs disposent des ressources nécessaires). Les corpus d'entraînement, largement issus du web, véhiculent des biais sociaux, culturels et linguistiques que les modèles reproduisent, voire amplifient. Enfin, l'évaluation de ces modèles reste un défi : les benchmarks statiques utilisés (GLUE, SQuAD, etc.) ne rendent pas toujours compte de la capacité réelle de raisonnement ou de la fiabilité factuelle des réponses produites.

**Pistes de recherche futures.** Les articles les plus récents (GPT-3, PaLM) appellent, en filigrane, à des travaux sur l'efficacité des modèles (distillation, quantification, architectures creuses) afin de réduire leur coût, sur l'alignement des modèles avec les intentions et valeurs humaines, sur l'extension à d'autres modalités (image, audio, code), et sur des méthodes d'évaluation plus robustes capables de mesurer le raisonnement, la factualité et les biais plutôt que la seule performance moyenne sur des benchmarks figés.

## 6. Conclusion

Cette méta-analyse montre que les cinq articles étudiés ne constituent pas une simple succession de records de performance, mais les étapes d'une même trajectoire scientifique cohérente. L'architecture Transformer a fourni le socle commun ; BERT et GPT ont exploré deux directions complémentaires (compréhension bidirectionnelle et génération autorégressive) ; GPT-3 et PaLM ont ensuite démontré que le passage à l'échelle de cette architecture transforme qualitativement les capacités des modèles, jusqu'à l'apprentissage en contexte et le raisonnement émergent. Cette trajectoire éclaire également les grands défis actuels du domaine — coût computationnel, biais, évaluation — qui continuent aujourd'hui de structurer la recherche sur les grands modèles de langage, bien au-delà des cinq articles ici analysés.